# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


In this exercise, we will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this exercise


In [2]:
import sys

import requests    
from bs4 import BeautifulSoup
import re    # Provides support for working with regular expressions to search, match, and manipulate text patterns
import unicodedata    # Provides access to the Unicode Character Database to look up and normalize character properties (like handling special spaces or symbols)
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


#### 1. `date_time(table_cells)`

**Purpose:** Extracts the **date** and **time** of the launch.

##### How It Works

- `table_cells.strings` retrieves all text contained within the HTML table cell, including text inside nested HTML tags.
- A **list comprehension** iterates through each string and removes leading and trailing whitespace using `.strip()`.
- `[0:2]` returns only the first two elements:
  - **Element 0:** Launch Date
  - **Element 1:** Launch Time

> **Result:** The function returns the launch **date** and **time**.

#### 2. `booster_version(table_cells)`

**Purpose:** Extracts the rocket's **booster version** while filtering out unwanted HTML content.

##### How It Works

- Iterates through `table_cells.strings` using `enumerate()` to obtain both:
  - The **index (`i`)**
  - The **text value**
- `if i % 2 == 0:` selects only the **even-indexed** text elements.
  - This conveniently skips unwanted intermediate elements such as `<br>` tags or alternating HTML nodes.
- `[0:-1]` removes the last element, which is typically:
  - A reference link
  - A footnote
  - Other unnecessary text
- `''.join(...)` concatenates the remaining text fragments into a single clean string.

> **Result:** A clean booster version string is returned.

#### 3. `landing_status(table_cells)`

**Purpose:** Extracts the **landing outcome** of the booster (e.g., *Success* or *Failure*).

##### How It Works

- Converts all text inside the HTML cell into a list.
- Returns the first element using:

```python
[0]
```

- The first string corresponds to the landing status.

> **Result:** Returns the booster landing status.

#### 4. `get_mass(table_cells)`

**Purpose:** Extracts the payload **mass** while removing unwanted characters and extra text.

##### How It Works

- `unicodedata.normalize("NFKD", table_cells.text)` converts Unicode characters into their standard text representation.
- Checks whether the extracted mass contains any value.
- `mass.find("kg") + 2` finds the ending position of the `"kg"` unit.
- `mass[0 : mass.find("kg") + 2]` extracts everything from the beginning of the string up to and including `"kg"`.
- If no mass value exists, the function returns **0**.

##### Example

| Original Text | Extracted Value |
|---------------|-----------------|
| `5,600 kg [12]` | `5,600 kg` |
| `13,150 kg¹` | `13,150 kg` |
| *(Empty)* | `0` |

> **Result:** Returns a clean payload mass value.

#### 5. `extract_column_from_header(row)`

**Purpose:** Cleans HTML table headers (`<th>`) to produce readable Python dictionary keys.

##### How It Works

The function removes unnecessary HTML elements:

| HTML Element | Purpose of Removal |
|--------------|--------------------|
| `<br>` | Removes line breaks |
| `<a>` | Removes hyperlinks |
| `<sup>` | Removes superscript footnotes |

It performs the following operations:

- `row.br.extract()` removes line breaks.
- `row.a.extract()` removes hyperlinks.
- `row.sup.extract()` removes superscript footnotes.
- `' '.join(row.contents)` combines the remaining text into a single string.
- `column_name.strip()` removes leading and trailing whitespace.
- `if not (column_name.strip().isdigit()):`
  - Checks whether the header consists only of numbers.
  - Numeric-only headers (typically footnotes or indices) are ignored.

> **Result:** Returns a clean, readable column name suitable for use as a Python dictionary key.

To keep the exercise tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

**headers**
- Purpose: Sets a custom User-Agent string for the web request.
- The Problem: The default Python identity ("python-requests") is often blocked or restricted by websites like Wikipedia to prevent automated spam.
- The Logic: This dictionary mimics a standard Google Chrome browser on Windows 10, tricking the server into treating your script like a normal human visitor so the request goes through smoothly.

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
# Perform an HTTP GET request
response = requests.get(static_url, headers=headers)

Create a `BeautifulSoup` object from the HTML `response`


In [6]:
# Use BeautifulSoup() to create a BeautifulSoup object from a response text content
soup = BeautifulSoup(response.text, 'html.parser')

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [7]:
# Print the page title to verify
print(soup.title)

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
# Find all table elements on the page
html_tables = soup.find_all('table')

Starting from the third table is our target table contains the actual launch records.


In [9]:
# Let's print the third table and check its content
first_launch_table = html_tables[2]
print(first_launch_table)

<table class="wikitable plainrowheaders collapsible" style="width: 100%;">
<tbody><tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11"><span class="cite-bracket">[</span>b<span class="cite-bracket">]</span></a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12"><span class="cite-bracket">[</span>c<span class="cite-bracket">]</span></a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
column_names = []

# Find all 'th' elements in the target table
for th in first_launch_table.find_all('th'):
    # Apply the helper function from cell 3
    name = extract_column_from_header(th)
    # Append to the list if it's not empty or None
    if name is not None and len(name) > 0:
        column_names = column_names + [name]

Check the extracted column names


In [11]:
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
launch_dict= dict.fromkeys(column_names)

# Remove an irrelvant column
del launch_dict['Date and time ( )']

# Let's initial the launch_dict with each value to be an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [13]:
extracted_row = 0
# Extract each table 
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
   # Get table row
    for rows in table.find_all("tr"):
        # Check to see if first table heading is as number corresponding to launch a flight 
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
            
        # Get table element 
        row = rows.find_all('td')
        
        # If it is number save cells in a dictionary 
        if flag:
            extracted_row += 1
            
            # Flight Number
            launch_dict['Flight No.'].append(flight_number)
            
            # Date and Time
            datatimelist = date_time(row[0])
            date = datatimelist[0].strip(',')
            time = datatimelist[1]
            launch_dict['Date'].append(date)
            launch_dict['Time'].append(time)
            
            # Booster version
            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a else None
            launch_dict['Version Booster'].append(bv)
            
            # Launch Site
            launch_site = row[2].a.string if row[2].a else None
            launch_dict['Launch site'].append(launch_site)
            
            # Payload
            payload = row[3].a.string if row[3].a else None
            launch_dict['Payload'].append(payload)
            
            # Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)
            
            # Orbit
            orbit = row[5].a.string if row[5].a else None
            launch_dict['Orbit'].append(orbit)
            
            # Customer
            customer = row[6].a.string if row[6].a else None
            launch_dict['Customer'].append(customer)
            
            # Launch Outcome
            launch_outcome = list(row[7].strings)[0] if row[7] else None
            launch_dict['Launch outcome'].append(launch_outcome)
            
            # Booster Landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

#### Part 1: Setting Up the Loops and Filters

##### Initialize the Row Counter

```python
extracted_row = 0
```

- **Keyword:** Variable assignment (`=`)
- **Logic:** Initializes a counter variable starting at **0**. This keeps track of how many valid, data-filled launch rows the script has successfully processed.

##### Loop Through All Target Tables

```python
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
```

- **Keywords:**
  - `for` – Definite loop
  - `enumerate()` – Index tracker
  - `soup.find_all()` – BeautifulSoup search method

- **Logic:**
  - `soup.find_all()` searches for all Wikipedia tables matching the CSS class:
    ```
    wikitable plainrowheaders collapsible
    ```
  - `enumerate()` assigns each table a sequential index (`table_number`).
  - The `for` loop processes each table one by one.

##### Loop Through Each Table Row

```python
for rows in table.find_all("tr"):
```

- **Keywords:**
  - `for` – Nested loop
  - `.find_all("tr")` – HTML table row lookup

- **Logic:**
  - Iterates through every horizontal table row (`<tr>`) within the current table from top to bottom.

---

#### Part 2: Checking for a Valid Launch Number

##### Validate the Flight Number

```python
if rows.th:
    if rows.th.string:
        flight_number = rows.th.string.strip()
        flag = flight_number.isdigit()
```

- **Keywords:**
  - `if`
  - `.th`
  - `.string`
  - `.strip()`
  - `.isdigit()`

- **Logic:**
  1. `if rows.th:` checks whether the row contains a table header (`<th>`).
  2. `if rows.th.string:` ensures the header contains plain text.
  3. `.strip()` removes any leading or trailing whitespace.
  4. `.isdigit()` verifies whether the text contains only digits (e.g., `"1"`, `"2"`).

- **Purpose:**
  - If the value is numeric, `flag = True`.
  - This filters out section headers and non-launch rows.

##### Handle Invalid Rows

```python
else:
    flag = False
```

- **Keyword:** `else`

- **Logic:**
  - If the row does not contain a `<th>` element, `flag` is set to `False`.
  - This prevents non-launch rows from being processed.

---

#### Part 3: Unpacking the Data Cells

##### Extract All Data Cells

```python
row = rows.find_all('td')
```

- **Keyword:** `.find_all('td')`

- **Logic:**
  - Extracts every table data cell (`<td>`) from the current row.
  - Stores them as a zero-indexed Python list named `row`.

##### Process Only Valid Launch Rows

```python
if flag:
    extracted_row += 1
```

- **Keywords:**
  - `if`
  - `+= 1`

- **Logic:**
  - Executes only if the row represents a valid launch.
  - Increments the processed launch counter by **1**.

---

#### Part 4: Extracting and Appending Row Values

##### Store the Flight Number

```python
launch_dict['Flight No.'].append(flight_number)
```

- **Keyword:** `.append()`

- **Logic:**
  - Appends the cleaned flight number (e.g., `"1"`) to the **Flight No.** list.

---

##### Extract the Launch Date and Time

```python
datatimelist = date_time(row[0])
date = datatimelist[0].strip(',')
time = datatimelist[1]

launch_dict['Date'].append(date)
launch_dict['Time'].append(time)
```

| Keyword | Purpose |
|---------|---------|
| `date_time()` | Custom helper function |
| `[0]`, `[1]` | List indexing |
| `.strip()` | Removes unwanted commas |
| `.append()` | Stores values |

**Logic:**

1. Passes the first data cell (`row[0]`) to the custom `date_time()` function.
2. Splits the returned value into:
   - **Date**
   - **Time**
3. Removes any trailing commas from the date.
4. Stores both values in their corresponding lists.

---

##### Extract the Booster Version

```python
bv = booster_version(row[1])

if not bv:
    bv = row[1].a.string if row[1].a else None

launch_dict['Version Booster'].append(bv)
```

- **Keywords:**
  - `booster_version()`
  - `if not`
  - `if/else`
  - `.a.string`

**Logic:**

1. Sends `row[1]` to the custom `booster_version()` function.
2. If the function returns nothing:
   - Checks whether an anchor (`<a>`) tag exists.
   - If it exists, extracts the hyperlink text.
   - Otherwise, assigns `None`.
3. Stores the final booster version in the dictionary.

---

##### Extract the Launch Site

```python
launch_site = row[2].a.string if row[2].a else None

launch_dict['Launch site'].append(launch_site)
```

- **Keywords:**
  - `if/else`
  - `.a.string`
  - `.append()`

**Logic:**

1. Checks whether the third data cell contains an anchor (`<a>`) tag.
2. If present:
   - Extracts the hyperlink text (e.g., `"CCAFS"`).
3. Otherwise:
   - Assigns `None`.
4. Saves the launch site to the dictionary.

---

##### Extract the Payload

```python
payload = row[3].a.string if row[3].a else None

launch_dict['Payload'].append(payload)
```

- **Keywords:**
  - `if/else`
  - `.a.string`
  - `.append()`

**Logic:**

1. Applies the same extraction method to the fourth data cell (`row[3]`).
2. Retrieves the payload name (e.g., `"Dragon Spacecraft Qualification Unit"`).
3. If no anchor tag exists, assigns `None`.
4. Appends the payload name to the **Payload** list.

After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [14]:
df= pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this exercise. 

Following exercises will be using a provided dataset to make each lab independent. 


In [15]:
df.to_csv('Assets/2_spacex_web_scraped.csv', index=False)